# 11 · Profile your own kernel

Nine case studies, one method. This notebook is the method, plus the traps that cost the most time to discover.

## 1. The method

**Start with three ranges.** Prologue, mainloop, epilogue. Do not instrument what you have not yet found to be interesting — notebook 01 found the answer with three ranges, and the answer was "look at the epilogue".

**Add detail only where the first profile points.** Notebook 02 needed per-iteration `cons_wait` and `prod_acquire` — but only after the coarse profile showed the mainloop mattered. Instrumenting everything up front is how you overflow the trace buffer and learn nothing.

**Instrument both ends of a pipeline.** The single most transferable finding in this series: relieving a stall at one end can create one at the other. Notebook 02's PS3 improved `cons_wait` and got slower, and only `prod_acquire` explained it.

**Compare variants, not single traces.** Every notebook that produced a decision — 02, 03, 05 — did it by diffing two runs that differed in one thing. A single trace tells you what happened; two tell you what to change.

**Name the mechanism you are testing.** `tma_clc_wait` exists because CLC exists. When the overhead of a mechanism has its own range, its cost is measured rather than inferred from a wall-clock delta.

## 2. Instrumentation choices that matter

### Marks for issues, ranges for waits

| Event | Use | Why |
|---|---|---|
| TMA issue, MMA issue, buffer release | `iket.mark` | An instant, not a duration. One event, not two. |
| Waits, phases, role blocks | `range_push`/`pop` | A real duration you want to measure |


The `_marks` style roughly halves event volume against annotating everything as ranges. That is the difference between a full trace and an empty one on a complex kernel.

### Warp-uniformity is a hard requirement

IKET events are **warp-level**. Instrument points must be warp-uniform. If threads within a warp diverge at an instrument point, the result is not an error — it is wrong pairings and an unpredictable number of dumps. Put instrumentation on warp-uniform control flow.

### Scope the capture

`--enabled-cluster 0,0,0` traces a single cluster, producing a trace smaller by roughly the grid factor. Use it for per-warp phase analysis. Use the full grid when the question is about occupancy, load imbalance, or cross-CTA overlap — as in notebooks 06 and 08.

## 3. Traps

In rough order of how much time they cost to discover.

### Trace-buffer overflow is silent

An over-instrumented kernel can capture **zero** events and report no error. It looks like a broken profiler. It is not — you asked for more events than the buffer holds.

Symptoms and fixes: no `.trace.json`, or one with no ranges. Reduce instrumentation density, switch async issues to marks, or add `--enabled-cluster 0,0,0`.

*Evidence this is real:* `fp16_gemm_3_1` previously captured nothing at all. Re-annotated in the `_marks` style, the same kernel captures 73,804 ranges and 14,848 markers.

### Graph-launched kernels hide in a different key

CUDA-graph-launched kernels put their events under `graphLaunches`, not `launches`. Read only `launches` and the trace looks empty — indistinguishable from overflow, but with a completely different cause. Use `trace_summary.all_launches()`.

### Timestamps are nanoseconds

Not clock cycles. Values landing on multiples of 32 are the tell. A 0.032 µs median means *one quantum* — at or below the resolution of the instrumentation, not a real duration you should reason about.

### Instrumentation density can invert a comparison

Adding instrumentation costs time. Comparing a heavily-instrumented variant against a lightly-instrumented one can reverse the ordering. Keep instrumentation **identical** across variants in an A/B, and sanity-check the conclusion against un-profiled wall-clock runs.

### A long wait is not automatically a problem

Notebook 04's `acc_wait` is the largest range in the kernel at 2.752 µs and is entirely by design — the epilogue warp idling in the mainloop's shadow. Ranking ranges by duration and attacking the top one will send you optimising a wait that should be long. Ask *which role* is waiting and *what for*.

## 4. What IKET gives you that kernel timing cannot

1. Validate that a kernel behaves the way the code claims (notebooks 01, 07)
2. Identify *which* wait is long, and which role is waiting (02, 04, 05)
3. See whether roles overlap as designed (04, 10)
4. Compare design variants as a subtraction rather than an interpretation (02, 03, 09)
5. Understand SM, CTA and cluster utilisation (06, 08)


And the one that matters most: **avoid false conclusions from kernel-level timing alone.** Notebook 08's PDL result was a 1.4% wall-clock difference that a benchmark would have shrugged at. The trace showed the mechanism worked perfectly and the binding constraint was elsewhere.

## 5. Where to go next

- **The reference guide** — [`iket_profiling.rst`](https://github.com/NVIDIA/cutlass/blob/main/media/docs/pythonDSL/guides/iket_profiling.rst) in CUTLASS covers the full API, payloads, pairing models, and the JSON schema
- **A worked upstream example** — `examples/python/CuTeDSL/dsl_tutorials/fp16_gemm_4_iket.py`
- **Hardware metrics** — `ncu --section SpeedOfLight` answers *why* once IKET has told you *where*


### Your first profile on your own kernel

```python
iket = cute.experimental.iket

token = iket.range_start("prologue")
# ... setup ...
iket.range_end(token)

iket.range_push("mainloop")
# ... the loop ...
iket.range_pop()

iket.range_push("epilogue")
# ... drain ...
iket.range_pop()
```

```bash
run-iket --output-dir out --clobber profile --postprocess all -- python your_kernel.py
```

Then open the `.pftrace` at [ui.perfetto.dev](https://ui.perfetto.dev/) and find out whether your kernel does what you think it does.

It usually does not. That is the point.

You have reached the end of the series.

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*